In [1]:
import asyncio
from sqlalchemy.ext.asyncio import AsyncSession, create_async_engine
from sqlalchemy.orm import declarative_base, sessionmaker

from sqlalchemy.future import select
from app.models.log import HttpRequestLog  # import your HttpRequestLog model
from contextlib import asynccontextmanager
from sqlalchemy.orm import Session
from typing import AsyncIterator, List

import os

class ProjectSettings:
    PROJECT_NAME: str = "Anomaly Detection"
    PROJECT_VERSION: str = "1.0.0"

    # Database settings
    POSTGRES_USER: str = os.getenv("POSTGRES_USER", "postgres")
    POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD", "postgres")
    POSTGRES_DB: str = os.getenv("POSTGRES_DB", "anomaly_detection")
    POSTGRES_SERVER: str = os.getenv("POSTGRES_SERVER", "127.0.0.1")
    POSTGRES_PORT: str = os.getenv("POSTGRES_PORT", 5432)  # default postgres port is 5432

    # Connection strings
    DATABASE_URL = f"postgresql+asyncpg://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_SERVER}:{POSTGRES_PORT}/{POSTGRES_DB}"
    SYNC_DB_URL = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_SERVER}:{POSTGRES_PORT}/{POSTGRES_DB}"

projectSettings = ProjectSettings()

engine = create_async_engine(projectSettings.DATABASE_URL, future=True, echo=True, pool_size=20, max_overflow=10)


async_session = sessionmaker(engine, expire_on_commit=False, class_=AsyncSession)

@asynccontextmanager
async def get_db() -> AsyncIterator[AsyncSession]:
    async with async_session() as session:
        yield session


async def fetch_data(db: Session) -> List[HttpRequestLog]:
    result = await db.execute(select(HttpRequestLog))
                              # .offset(skip).limit(limit))
    return result.scalars().all()

# Usage
# data = asyncio.run(fetch_data())
# Instead of asyncio.run(fetch_data()), directly await the coroutine
async with get_db() as db:
    data = await fetch_data(db)


data[0]

2024-01-16 17:32:30,292 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2024-01-16 17:32:30,296 INFO sqlalchemy.engine.Engine [raw sql] ()
2024-01-16 17:32:30,309 INFO sqlalchemy.engine.Engine select current_schema()
2024-01-16 17:32:30,309 INFO sqlalchemy.engine.Engine [raw sql] ()
2024-01-16 17:32:30,310 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2024-01-16 17:32:30,311 INFO sqlalchemy.engine.Engine [raw sql] ()
2024-01-16 17:32:30,314 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2024-01-16 17:32:30,316 INFO sqlalchemy.engine.Engine SELECT logs.id, logs.created_at, logs.updated_at, logs.client_host, logs.request_time, logs.request_method, logs.request_path, logs.http_version, logs.referrer, logs.user_agent, logs.user_id, logs.log_message, logs.cookies, logs.post_params, logs.get_params, logs.body, logs.response_status, logs.process_time 
FROM logs
2024-01-16 17:32:30,316 INFO sqlalchemy.engine.Engine [generated in 0.00036s] ()
2024-01-16 17:32:30,512 I

In [2]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import json
import httpagentparser

# Assuming data is a list of HttpRequestLog instances
df = pd.DataFrame([record.__dict__ for record in data])
df = df.drop(['_sa_instance_state'], axis=1)

# Handling JSON data
json_features = ['cookies', 'post_params', 'get_params']

for feature in json_features:
    df[feature] = df[feature].apply(lambda x: json.loads(x) if pd.notna(x) else {})
    df[f"{feature}_count"] = df[feature].apply(lambda x: len(x.keys()))

# Handling textual data (e.g., user_agent)
df['browser'] = df['user_agent'].apply(lambda x: httpagentparser.detect(x)['browser']['name'] if httpagentparser.detect(x).get('browser') else 'Unknown')

# One-Hot Encoding
one_hot_encoder = OneHotEncoder(handle_unknown='ignore')
categorical_features = ['request_method', 'http_version', 'user_id', 'browser']

# Fit and transform the categorical features
encoded_features = one_hot_encoder.fit_transform(df[categorical_features]).toarray()
columns = one_hot_encoder.get_feature_names_out(categorical_features)
df_encoded = pd.DataFrame(encoded_features, columns=columns, index=df.index)

# Concatenate the encoded features with the original DataFrame
df = pd.concat([df, df_encoded], axis=1)

# Drop the original categorical and JSON columns, as they are now encoded
fields_to_drop = categorical_features + json_features + ['id','body','user_agent', 'referrer', 'log_message', 'body', 'created_at', 'updated_at', 'client_host', 'request_path']
train_df = df.drop(columns=fields_to_drop)

train_df[['process_time', 'response_status']] = train_df[['response_status', 'process_time']]
train_df = train_df.rename(columns={'process_time': 'response_status', 'response_status': 'process_time'})

# Your DataFrame is now preprocessed and ready for use
train_df

,request_time,response_status,process_time,cookies_count,post_params_count,get_params_count,request_method_GET,request_method_POST,http_version_HTTP/1.0,user_id_6.0,...,user_id_10.0,user_id_14.0,user_id_16.0,user_id_18.0,user_id_19.0,user_id_nan,browser_Chrome,browser_Firefox,browser_Safari,browser_Unknown
0,1.630213e+09,200,0.0,1,0,0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
1,1.630213e+09,302,134.0,1,0,0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
2,1.630213e+09,200,0.0,1,0,1,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
3,1.630213e+09,302,13.0,1,3,1,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,1.630213e+09,302,0.0,2,0,0,1.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8086,1.630334e+09,302,1797129.0,1,4,0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
8087,1.630334e+09,302,1797130.0,1,0,0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
8088,1.630371e+09,302,1760326.0,1,0,0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
8089,1.630371e+09,302,1760327.0,1,0,0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


In [166]:
train_df.columns

Index(['request_time', 'response_status', 'process_time', 'cookies_count',
       'post_params_count', 'get_params_count', 'request_method_GET',
       'request_method_POST', 'http_version_HTTP/1.0', 'user_id_6.0',
       'user_id_9.0', 'user_id_10.0', 'user_id_14.0', 'user_id_16.0',
       'user_id_18.0', 'user_id_19.0', 'user_id_nan', 'browser_Chrome',
       'browser_Firefox', 'browser_Safari', 'browser_Unknown'],
      dtype='object')

In [4]:
one_hot_encoder.fit(df[categorical_features])


OneHotEncoder(handle_unknown='ignore')

In [5]:
one_hot_encoder

OneHotEncoder(handle_unknown='ignore')

In [8]:
import joblib
joblib.dump(one_hot_encoder, 'models/one_hot_encoder.joblib')


['models/one_hot_encoder.joblib']

In [167]:
from sklearn.ensemble import IsolationForest

# Model Training
def train_model(df):
    model = IsolationForest(n_estimators=100, contamination=0.05)
    model.fit(df)
    return model

In [168]:
# Train the model
model = train_model(train_df)

/Users/roviso/opt/anaconda3/envs/py38/lib/python3.8/site-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


In [188]:
import joblib

# Save the model
joblib.dump(model, 'models/IsolationForest_model.pkl')

['models/IsolationForest_model.pkl']

In [9]:
import pandas as pd
import json
import httpagentparser

# Assuming one_hot_encoder is the OneHotEncoder instance fitted during training
global one_hot_encoder

def inference_preprocess_data(log_instance):
    log_data = log_instance.__dict__

    # Handle JSON fields
    json_features = ['cookies', 'post_params', 'get_params']
    for field in json_features:
        log_data[field] = json.loads(log_data[field]) if log_data[field] else {}
        log_data[f"{field}_count"] = len(log_data[field].keys())

    # Handling textual data (e.g., user_agent)
    log_data['browser'] = httpagentparser.detect(log_data['user_agent'])['browser']['name'] if httpagentparser.detect(log_data['user_agent']).get('browser') else 'Unknown'

    # Prepare DataFrame for one-hot encoding
    df = pd.DataFrame([log_data])
    df = df.drop(['_sa_instance_state'], axis=1)

    
    # Fill missing categorical columns with 'Unknown'
    categorical_features = ['request_method', 'http_version', 'user_id', 'browser']
    for feature in categorical_features:
        if feature not in df.columns:
            df[feature] = 'Unknown'

    # Apply one-hot encoding to the categorical features
    encoded_df = pd.DataFrame(one_hot_encoder.transform(df[categorical_features]).toarray(), columns=one_hot_encoder.get_feature_names_out())

    # Concatenate encoded features with the rest of the data
    df = pd.concat([df.drop(columns=categorical_features), encoded_df], axis=1)

    # Drop the original categorical and JSON columns as they are now encoded
    fields_to_drop = json_features + ['user_agent', 'referrer', 'log_message', 'body', 'client_host', 'request_path']
    df = df.drop(columns=fields_to_drop)

    return df


In [10]:
def infer_anomaly(model, log_string):
    log_instance = create_http_request_log_instance_from_string(log_string)
    if log_instance is None:
        return "Invalid log format"

    df = inference_preprocess_data(log_instance)
    prediction = model.predict(df)

    if prediction[0] == -1:
        return "Anomalous log detected"
    else:
        return "Normal log"


In [11]:
import re
from app.models.log import HttpRequestLog  # Import your HttpRequestLog model

def get_log_data(log):
    re_exp = '(^[0-9]+\.[0-9]+\.[0-9]+\.[0-9]+) ([0-9]*) \"(.*?) (.*?)\" (.*?) (.*?) \"(.*?)\" \"(.*?)\" \"(.*?)\" \"(.*?)\" (.*?) ({.*?}) ({.*?}) ({.*?})'
    print(log,"log")
    match = re.search(re_exp, log)
    if not match:
        print("NO MATCH FOUNND")
        return None

    return {
        'client_host': match.group(1),
        'request_time': float(match.group(2)) if match.group(2) else None,
        'request_method': match.group(3),
        'request_path': match.group(4),
        'http_version': match.group(5),
        'response_status': int(match.group(6).split()[0]) if match.group(6).split() else None,
        'process_time': float(match.group(6).split()[1]) if len(match.group(6).split()) > 1 else None,
        'referrer': match.group(7),
        'user_agent': match.group(8),
        'user_id': int(match.group(10)) if match.group(10).isdigit() else None,
        'log_message': match.group(11),
        'cookies': match.group(12),
        'post_params': match.group(13),
        'get_params': match.group(14),
        'body': None
    }

def create_http_request_log_instance_from_string(log_string):
    log_data = get_log_data(log_string)
    if log_data is None:
        return None  # Or handle the error as per your application's need
    print(log_data,111)
    # Create an instance of HttpRequestLog with the extracted data
    log_instance = HttpRequestLog(
        client_host=log_data['client_host'],
        request_time=log_data['request_time'],
        request_method=log_data['request_method'],
        request_path=log_data['request_path'],
        http_version=log_data['http_version'],
        response_status=log_data['response_status'],
        process_time=log_data['process_time'],
        referrer=log_data['referrer'],
        user_agent=log_data['user_agent'],
        user_id=log_data['user_id'],
        log_message=log_data['log_message'],
        cookies=log_data['cookies'],
        post_params=log_data['post_params'],
        get_params=log_data['get_params'],
        body=log_data['body']
    )

    return log_instance


In [172]:
log_string = """8.20.101.59 1630213368 "POST /dashboard/users/create" HTTP/1.0 302 32 "http://localhost:3000/dashboard/users/create" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36" "-" "6" - {"fastapi-csrf-token": "xwHvkU1SzNqx6S22sPHgJ1qFy3ST7GBSgBdOa7ALsyy5ycdIFcmjTbfvRhRYedFP", "sessionId": "9nrhkxte0z38n8k7bfjkyi2v1zgbxu3h"} {"csrfmiddlewaretoken": "3Q4OMEsz7icX5dIsQ5AxBgYYVaO6lB2rMVA7CR1s03kvxxT83sfALqNOeoNbs86o", "username": "aashutosh", "email": "aashutosh9881@gmail.com", "hospital": "5"} {}"""


In [173]:
log_instance = create_http_request_log_instance_from_string(log_string)

df = inference_preprocess_data(log_instance)
df

8.20.101.59 1630213368 "POST /dashboard/users/create" HTTP/1.0 302 32 "http://localhost:3000/dashboard/users/create" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36" "-" "6" - {"fastapi-csrf-token": "xwHvkU1SzNqx6S22sPHgJ1qFy3ST7GBSgBdOa7ALsyy5ycdIFcmjTbfvRhRYedFP", "sessionId": "9nrhkxte0z38n8k7bfjkyi2v1zgbxu3h"} {"csrfmiddlewaretoken": "3Q4OMEsz7icX5dIsQ5AxBgYYVaO6lB2rMVA7CR1s03kvxxT83sfALqNOeoNbs86o", "username": "aashutosh", "email": "aashutosh9881@gmail.com", "hospital": "5"} {} log
{'client_host': '8.20.101.59', 'request_time': 1630213368.0, 'request_method': 'POST', 'request_path': '/dashboard/users/create', 'http_version': 'HTTP/1.0', 'response_status': 302, 'process_time': 32.0, 'referrer': 'http://localhost:3000/dashboard/users/create', 'user_agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36', 'user_id': 6, 'log_message': 

,request_time,response_status,process_time,cookies_count,post_params_count,get_params_count,request_method_GET,request_method_POST,http_version_HTTP/1.0,user_id_6.0,...,user_id_10.0,user_id_14.0,user_id_16.0,user_id_18.0,user_id_19.0,user_id_nan,browser_Chrome,browser_Firefox,browser_Safari,browser_Unknown
0,1.630213e+09,302,32.0,2,4,0,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [174]:
train_df

,request_time,response_status,process_time,cookies_count,post_params_count,get_params_count,request_method_GET,request_method_POST,http_version_HTTP/1.0,user_id_6.0,...,user_id_10.0,user_id_14.0,user_id_16.0,user_id_18.0,user_id_19.0,user_id_nan,browser_Chrome,browser_Firefox,browser_Safari,browser_Unknown
0,1.630213e+09,200,0.0,1,0,0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
1,1.630213e+09,302,134.0,1,0,0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
2,1.630213e+09,200,0.0,1,0,1,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
3,1.630213e+09,302,13.0,1,3,1,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,1.630213e+09,302,0.0,2,0,0,1.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8086,1.630334e+09,302,1797129.0,1,4,0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
8087,1.630334e+09,302,1797130.0,1,0,0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
8088,1.630371e+09,302,1760326.0,1,0,0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
8089,1.630371e+09,302,1760327.0,1,0,0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


In [175]:
one_hot_encoder.get_feature_names_out()

array(['request_method_GET', 'request_method_POST',
       'http_version_HTTP/1.0', 'user_id_6.0', 'user_id_9.0',
       'user_id_10.0', 'user_id_14.0', 'user_id_16.0', 'user_id_18.0',
       'user_id_19.0', 'user_id_nan', 'browser_Chrome', 'browser_Firefox',
       'browser_Safari', 'browser_Unknown'], dtype=object)

In [176]:
example_log = """8.20.101.59 1630213368 "POST /dashboard/users/create" HTTP/1.0 302 32 "http://localhost:3000/dashboard/users/create" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36" "-" "6" - {"fastapi-csrf-token": "xwHvkU1SzNqx6S22sPHgJ1qFy3ST7GBSgBdOa7ALsyy5ycdIFcmjTbfvRhRYedFP", "sessionId": "9nrhkxte0z38n8k7bfjkyi2v1zgbxu3h"} {"csrfmiddlewaretoken": "3Q4OMEsz7icX5dIsQ5AxBgYYVaO6lB2rMVA7CR1s03kvxxT83sfALqNOeoNbs86o", "username": "aashutosh", "email": "aashutosh9881@gmail.com", "hospital": "5"} {}"""

result = infer_anomaly(model, example_log)
print(result)


8.20.101.59 1630213368 "POST /dashboard/users/create" HTTP/1.0 302 32 "http://localhost:3000/dashboard/users/create" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36" "-" "6" - {"fastapi-csrf-token": "xwHvkU1SzNqx6S22sPHgJ1qFy3ST7GBSgBdOa7ALsyy5ycdIFcmjTbfvRhRYedFP", "sessionId": "9nrhkxte0z38n8k7bfjkyi2v1zgbxu3h"} {"csrfmiddlewaretoken": "3Q4OMEsz7icX5dIsQ5AxBgYYVaO6lB2rMVA7CR1s03kvxxT83sfALqNOeoNbs86o", "username": "aashutosh", "email": "aashutosh9881@gmail.com", "hospital": "5"} {} log
{'client_host': '8.20.101.59', 'request_time': 1630213368.0, 'request_method': 'POST', 'request_path': '/dashboard/users/create', 'http_version': 'HTTP/1.0', 'response_status': 302, 'process_time': 32.0, 'referrer': 'http://localhost:3000/dashboard/users/create', 'user_agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36', 'user_id': 6, 'log_message': 

In [177]:
example_log = """110.44.127.184 1630333830 "GET /dashboard/accounts/login/" HTTP/1.0 200 1787719 "-" "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" - {} {} {} 
"""

result = infer_anomaly(model, example_log)
print(result)

110.44.127.184 1630333830 "GET /dashboard/accounts/login/" HTTP/1.0 200 1787719 "-" "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" - {} {} {} 
 log
{'client_host': '110.44.127.184', 'request_time': 1630333830.0, 'request_method': 'GET', 'request_path': '/dashboard/accounts/login/', 'http_version': 'HTTP/1.0', 'response_status': 200, 'process_time': 1787719.0, 'referrer': '-', 'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:92.0) Gecko/20100101 Firefox/92.0', 'user_id': None, 'log_message': '-', 'cookies': '{}', 'post_params': '{}', 'get_params': '{}', 'body': None} 111
Normal log


In [178]:
example_log = '''103.235.197.252 1630333830 "POST /dashboard/hospital/services/create" HTTP/1.0 302 1957984 "-" "python-requests/2.25.1" "-" "-" - {"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"} {"available_icu_beds": "2", "available_ventilators": "2", "available_isolation_beds": "2", "csrfmiddlewaretoken": "jbf3IhTNOuFV4c8elv6oZ1vDmJU2UufyURHMB45FUl88NEKIU6aakPwOandB2RPG"} {} 
'''
result = infer_anomaly(model, example_log)
print(result)

103.235.197.252 1630333830 "POST /dashboard/hospital/services/create" HTTP/1.0 302 1957984 "-" "python-requests/2.25.1" "-" "-" - {"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"} {"available_icu_beds": "2", "available_ventilators": "2", "available_isolation_beds": "2", "csrfmiddlewaretoken": "jbf3IhTNOuFV4c8elv6oZ1vDmJU2UufyURHMB45FUl88NEKIU6aakPwOandB2RPG"} {} 
 log
{'client_host': '103.235.197.252', 'request_time': 1630333830.0, 'request_method': 'POST', 'request_path': '/dashboard/hospital/services/create', 'http_version': 'HTTP/1.0', 'response_status': 302, 'process_time': 1957984.0, 'referrer': '-', 'user_agent': 'python-requests/2.25.1', 'user_id': None, 'log_message': '-', 'cookies': '{"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"}', 'post_params': '{"available_icu_beds": "2", "available_ventilators": "2", "available

In [179]:
example_log = '''110.44.127.184 1631776695 "POST /dashboard/accounts/login/" HTTP/1.0 200 442141 "http://logcovid.prixa.live/dashboard/accounts/login/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" __all__Incorrect username or password {"csrftoken": "FVP3hXV2nyOLBMStc6ygxpBcFyT9DZtUArYXbFvAgQ1IYReEHh7cVHoGTrI6Cl9w"} {"csrfmiddlewaretoken": "PWq66f5eR4tzNkP4n4GJnxmj4Bdq3Bd2Ksz00XFMKmGwapbfSffFLP9Niu2n2XTE", "username": "sdf", "password": "sdf))) OR ROW(2592,4840)>(SELECT COUNT(*),CONCAT(0x7162766b71,(SELECT (ELT(2592=2592,1))),0x716a767671,FLOOR(RAND(0)*2))x FROM (SELECT 8763 UNION SELECT 3269 UNION SELECT 7213 UNION SELECT 4244)a GROUP BY x) AND (((4527=4527"} {} 
'''
result = infer_anomaly(model, example_log)
print(result)

110.44.127.184 1631776695 "POST /dashboard/accounts/login/" HTTP/1.0 200 442141 "http://logcovid.prixa.live/dashboard/accounts/login/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" __all__Incorrect username or password {"csrftoken": "FVP3hXV2nyOLBMStc6ygxpBcFyT9DZtUArYXbFvAgQ1IYReEHh7cVHoGTrI6Cl9w"} {"csrfmiddlewaretoken": "PWq66f5eR4tzNkP4n4GJnxmj4Bdq3Bd2Ksz00XFMKmGwapbfSffFLP9Niu2n2XTE", "username": "sdf", "password": "sdf))) OR ROW(2592,4840)>(SELECT COUNT(*),CONCAT(0x7162766b71,(SELECT (ELT(2592=2592,1))),0x716a767671,FLOOR(RAND(0)*2))x FROM (SELECT 8763 UNION SELECT 3269 UNION SELECT 7213 UNION SELECT 4244)a GROUP BY x) AND (((4527=4527"} {} 
 log
{'client_host': '110.44.127.184', 'request_time': 1631776695.0, 'request_method': 'POST', 'request_path': '/dashboard/accounts/login/', 'http_version': 'HTTP/1.0', 'response_status': 200, 'process_time': 442141.0, 'referrer': 'http://logcovid.prixa.live/dashboard/accounts/login/', 'user_agen

In [13]:
from sklearn.neighbors import LocalOutlierFactor

# Assuming df is your preprocessed training DataFrame
lof_model = LocalOutlierFactor(n_neighbors=20, contamination=0.1,novelty=True)
# LOF doesn't have a .fit() method, it directly predicts on the data
y_pred = lof_model.fit(train_df)


In [14]:
import joblib

# Save the model
joblib.dump(lof_model, 'models/lof_model.pkl')


['models/lof_model.pkl']

In [185]:
def infer_anomaly(model_path, log_string):
    # Create a HttpRequestLog instance from the log string
    log_instance = create_http_request_log_instance_from_string(log_string)
    if log_instance is None:
        return "Invalid log format"

    # Preprocess the data
    df = inference_preprocess_data(log_instance)

    # Load the trained model
    model = joblib.load(model_path)

    # As LOF needs the training data, concatenate the new instance with the training data
    # Assuming you have the training data DataFrame `train_df` stored or can load it
    combined_df = pd.concat([train_df, df])

    # Predict using LOF model
    prediction = model.fit_predict(combined_df)

    # The prediction for the new data will be the last element
    if prediction[-1] == -1:
        return "Anomalous log detected"
    else:
        return "Normal log"


In [192]:
example_log = '''110.44.127.184 1631776695 "POST /dashboard/accounts/login/" HTTP/1.0 200 442141 "http://logcovid.prixa.live/dashboard/accounts/login/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" __all__Incorrect username or password {"csrftoken": "FVP3hXV2nyOLBMStc6ygxpBcFyT9DZtUArYXbFvAgQ1IYReEHh7cVHoGTrI6Cl9w"} {"csrfmiddlewaretoken": "PWq66f5eR4tzNkP4n4GJnxmj4Bdq3Bd2Ksz00XFMKmGwapbfSffFLP9Niu2n2XTE", "username": "sdf", "password": "sdf))) OR ROW(2592,4840)>(SELECT COUNT(*),CONCAT(0x7162766b71,(SELECT (ELT(2592=2592,1))),0x716a767671,FLOOR(RAND(0)*2))x FROM (SELECT 8763 UNION SELECT 3269 UNION SELECT 7213 UNION SELECT 4244)a GROUP BY x) AND (((4527=4527"} {} 
'''
result = infer_anomaly("models/lof_model.pkl", example_log)
print(result)
print(result,"\n\n\n")
result = infer_anomaly("models/IsolationForest_model.pkl", example_log)
print(result)

110.44.127.184 1631776695 "POST /dashboard/accounts/login/" HTTP/1.0 200 442141 "http://logcovid.prixa.live/dashboard/accounts/login/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" __all__Incorrect username or password {"csrftoken": "FVP3hXV2nyOLBMStc6ygxpBcFyT9DZtUArYXbFvAgQ1IYReEHh7cVHoGTrI6Cl9w"} {"csrfmiddlewaretoken": "PWq66f5eR4tzNkP4n4GJnxmj4Bdq3Bd2Ksz00XFMKmGwapbfSffFLP9Niu2n2XTE", "username": "sdf", "password": "sdf))) OR ROW(2592,4840)>(SELECT COUNT(*),CONCAT(0x7162766b71,(SELECT (ELT(2592=2592,1))),0x716a767671,FLOOR(RAND(0)*2))x FROM (SELECT 8763 UNION SELECT 3269 UNION SELECT 7213 UNION SELECT 4244)a GROUP BY x) AND (((4527=4527"} {} 
 log
{'client_host': '110.44.127.184', 'request_time': 1631776695.0, 'request_method': 'POST', 'request_path': '/dashboard/accounts/login/', 'http_version': 'HTTP/1.0', 'response_status': 200, 'process_time': 442141.0, 'referrer': 'http://logcovid.prixa.live/dashboard/accounts/login/', 'user_agen

/Users/roviso/opt/anaconda3/envs/py38/lib/python3.8/site-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


Anomalous log detected


In [191]:
example_log = '''110.44.127.184 1630333830 "GET /dashboard/accounts/login/" HTTP/1.0 200 1787719 "-" "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" - {} {} {} 
'''
result = infer_anomaly("models/lof_model.pkl", example_log)
print(result,"\n\n\n")
result = infer_anomaly("models/IsolationForest_model.pkl", example_log)
print(result)

110.44.127.184 1630333830 "GET /dashboard/accounts/login/" HTTP/1.0 200 1787719 "-" "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" - {} {} {} 
 log
{'client_host': '110.44.127.184', 'request_time': 1630333830.0, 'request_method': 'GET', 'request_path': '/dashboard/accounts/login/', 'http_version': 'HTTP/1.0', 'response_status': 200, 'process_time': 1787719.0, 'referrer': '-', 'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:92.0) Gecko/20100101 Firefox/92.0', 'user_id': None, 'log_message': '-', 'cookies': '{}', 'post_params': '{}', 'get_params': '{}', 'body': None} 111
Normal log 



110.44.127.184 1630333830 "GET /dashboard/accounts/login/" HTTP/1.0 200 1787719 "-" "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" - {} {} {} 
 log
{'client_host': '110.44.127.184', 'request_time': 1630333830.0, 'request_method': 'GET', 'request_path': '/dashboard/accounts/login/', 'http_version': 'HTTP/1.0', 'r

/Users/roviso/opt/anaconda3/envs/py38/lib/python3.8/site-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


Normal log


In [193]:
from sklearn.svm import OneClassSVM

# Assuming df is your preprocessed training DataFrame
oc_svm_model = OneClassSVM(gamma='auto', nu=0.05)
oc_svm_model.fit(train_df)


OneClassSVM(gamma='auto', nu=0.05)

In [194]:
import joblib

# Save the model
joblib.dump(oc_svm_model, 'models/oc_svm_model.pkl')


['models/oc_svm_model.pkl']

In [197]:
example_log = '''110.44.127.184 1631776695 "POST /dashboard/accounts/login/" HTTP/1.0 200 442141 "http://logcovid.prixa.live/dashboard/accounts/login/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" __all__Incorrect username or password {"csrftoken": "FVP3hXV2nyOLBMStc6ygxpBcFyT9DZtUArYXbFvAgQ1IYReEHh7cVHoGTrI6Cl9w"} {"csrfmiddlewaretoken": "PWq66f5eR4tzNkP4n4GJnxmj4Bdq3Bd2Ksz00XFMKmGwapbfSffFLP9Niu2n2XTE", "username": "sdf", "password": "sdf))) OR ROW(2592,4840)>(SELECT COUNT(*),CONCAT(0x7162766b71,(SELECT (ELT(2592=2592,1))),0x716a767671,FLOOR(RAND(0)*2))x FROM (SELECT 8763 UNION SELECT 3269 UNION SELECT 7213 UNION SELECT 4244)a GROUP BY x) AND (((4527=4527"} {} 
'''
result = infer_anomaly("models/lof_model.pkl", example_log)
print(result)
print("\n\n\n")
result = infer_anomaly("models/IsolationForest_model.pkl", example_log)
print(result)
print("\n\n\n")
result = infer_anomaly("models/oc_svm_model.pkl", example_log)
print(result)

110.44.127.184 1631776695 "POST /dashboard/accounts/login/" HTTP/1.0 200 442141 "http://logcovid.prixa.live/dashboard/accounts/login/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" __all__Incorrect username or password {"csrftoken": "FVP3hXV2nyOLBMStc6ygxpBcFyT9DZtUArYXbFvAgQ1IYReEHh7cVHoGTrI6Cl9w"} {"csrfmiddlewaretoken": "PWq66f5eR4tzNkP4n4GJnxmj4Bdq3Bd2Ksz00XFMKmGwapbfSffFLP9Niu2n2XTE", "username": "sdf", "password": "sdf))) OR ROW(2592,4840)>(SELECT COUNT(*),CONCAT(0x7162766b71,(SELECT (ELT(2592=2592,1))),0x716a767671,FLOOR(RAND(0)*2))x FROM (SELECT 8763 UNION SELECT 3269 UNION SELECT 7213 UNION SELECT 4244)a GROUP BY x) AND (((4527=4527"} {} 
 log
{'client_host': '110.44.127.184', 'request_time': 1631776695.0, 'request_method': 'POST', 'request_path': '/dashboard/accounts/login/', 'http_version': 'HTTP/1.0', 'response_status': 200, 'process_time': 442141.0, 'referrer': 'http://logcovid.prixa.live/dashboard/accounts/login/', 'user_agen

/Users/roviso/opt/anaconda3/envs/py38/lib/python3.8/site-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


Anomalous log detected




110.44.127.184 1631776695 "POST /dashboard/accounts/login/" HTTP/1.0 200 442141 "http://logcovid.prixa.live/dashboard/accounts/login/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" __all__Incorrect username or password {"csrftoken": "FVP3hXV2nyOLBMStc6ygxpBcFyT9DZtUArYXbFvAgQ1IYReEHh7cVHoGTrI6Cl9w"} {"csrfmiddlewaretoken": "PWq66f5eR4tzNkP4n4GJnxmj4Bdq3Bd2Ksz00XFMKmGwapbfSffFLP9Niu2n2XTE", "username": "sdf", "password": "sdf))) OR ROW(2592,4840)>(SELECT COUNT(*),CONCAT(0x7162766b71,(SELECT (ELT(2592=2592,1))),0x716a767671,FLOOR(RAND(0)*2))x FROM (SELECT 8763 UNION SELECT 3269 UNION SELECT 7213 UNION SELECT 4244)a GROUP BY x) AND (((4527=4527"} {} 
 log
{'client_host': '110.44.127.184', 'request_time': 1631776695.0, 'request_method': 'POST', 'request_path': '/dashboard/accounts/login/', 'http_version': 'HTTP/1.0', 'response_status': 200, 'process_time': 442141.0, 'referrer': 'http://logcovid.prixa.live/dashboard/a

In [198]:
example_log = '''110.44.127.184 1630333830 "GET /dashboard/accounts/login/" HTTP/1.0 200 1787719 "-" "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" - {} {} {} 
'''
result = infer_anomaly("models/lof_model.pkl", example_log)
print(result)
print("\n\n\n")
result = infer_anomaly("models/IsolationForest_model.pkl", example_log)
print(result)
print("\n\n\n")
result = infer_anomaly("models/oc_svm_model.pkl", example_log)
print(result)

110.44.127.184 1630333830 "GET /dashboard/accounts/login/" HTTP/1.0 200 1787719 "-" "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" - {} {} {} 
 log
{'client_host': '110.44.127.184', 'request_time': 1630333830.0, 'request_method': 'GET', 'request_path': '/dashboard/accounts/login/', 'http_version': 'HTTP/1.0', 'response_status': 200, 'process_time': 1787719.0, 'referrer': '-', 'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:92.0) Gecko/20100101 Firefox/92.0', 'user_id': None, 'log_message': '-', 'cookies': '{}', 'post_params': '{}', 'get_params': '{}', 'body': None} 111
Normal log




110.44.127.184 1630333830 "GET /dashboard/accounts/login/" HTTP/1.0 200 1787719 "-" "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" - {} {} {} 
 log
{'client_host': '110.44.127.184', 'request_time': 1630333830.0, 'request_method': 'GET', 'request_path': '/dashboard/accounts/login/', 'http_version': 'HTTP/1.0', 'r

/Users/roviso/opt/anaconda3/envs/py38/lib/python3.8/site-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


Normal log




110.44.127.184 1630333830 "GET /dashboard/accounts/login/" HTTP/1.0 200 1787719 "-" "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" - {} {} {} 
 log
{'client_host': '110.44.127.184', 'request_time': 1630333830.0, 'request_method': 'GET', 'request_path': '/dashboard/accounts/login/', 'http_version': 'HTTP/1.0', 'response_status': 200, 'process_time': 1787719.0, 'referrer': '-', 'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:92.0) Gecko/20100101 Firefox/92.0', 'user_id': None, 'log_message': '-', 'cookies': '{}', 'post_params': '{}', 'get_params': '{}', 'body': None} 111
Anomalous log detected


In [199]:
example_log = '''103.235.197.252 1631776695 "POST /dashboard/hospital/services/create" HTTP/1.0 302 513982 "-" "python-requests/2.25.1" "-" "-" - {"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"} {"available_icu_beds": "2", "available_ventilators": "2", "available_isolation_beds": "2", "csrfmiddlewaretoken": "jbf3IhTNOuFV4c8elv6oZ1vDmJU2UufyURHMB45FUl88NEKIU6aakPwOandB2RPG"} {} 
'''
result = infer_anomaly("models/lof_model.pkl", example_log)
print(result)
print("\n\n\n")
result = infer_anomaly("models/IsolationForest_model.pkl", example_log)
print(result)
print("\n\n\n")
result = infer_anomaly("models/oc_svm_model.pkl", example_log)
print(result)

103.235.197.252 1631776695 "POST /dashboard/hospital/services/create" HTTP/1.0 302 513982 "-" "python-requests/2.25.1" "-" "-" - {"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"} {"available_icu_beds": "2", "available_ventilators": "2", "available_isolation_beds": "2", "csrfmiddlewaretoken": "jbf3IhTNOuFV4c8elv6oZ1vDmJU2UufyURHMB45FUl88NEKIU6aakPwOandB2RPG"} {} 
 log
{'client_host': '103.235.197.252', 'request_time': 1631776695.0, 'request_method': 'POST', 'request_path': '/dashboard/hospital/services/create', 'http_version': 'HTTP/1.0', 'response_status': 302, 'process_time': 513982.0, 'referrer': '-', 'user_agent': 'python-requests/2.25.1', 'user_id': None, 'log_message': '-', 'cookies': '{"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"}', 'post_params': '{"available_icu_beds": "2", "available_ventilators": "2", "available_i

/Users/roviso/opt/anaconda3/envs/py38/lib/python3.8/site-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


Anomalous log detected




103.235.197.252 1631776695 "POST /dashboard/hospital/services/create" HTTP/1.0 302 513982 "-" "python-requests/2.25.1" "-" "-" - {"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"} {"available_icu_beds": "2", "available_ventilators": "2", "available_isolation_beds": "2", "csrfmiddlewaretoken": "jbf3IhTNOuFV4c8elv6oZ1vDmJU2UufyURHMB45FUl88NEKIU6aakPwOandB2RPG"} {} 
 log
{'client_host': '103.235.197.252', 'request_time': 1631776695.0, 'request_method': 'POST', 'request_path': '/dashboard/hospital/services/create', 'http_version': 'HTTP/1.0', 'response_status': 302, 'process_time': 513982.0, 'referrer': '-', 'user_agent': 'python-requests/2.25.1', 'user_id': None, 'log_message': '-', 'cookies': '{"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"}', 'post_params': '{"available_icu_beds": "2", "available_vent

In [202]:
import torch
import torch.nn as nn
import numpy as np

class Autoencoder(nn.Module):
    def __init__(self, input_size):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16)
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_size),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x


In [212]:
# Assume df is your preprocessed DataFrame
# Convert df to PyTorch tensor
data_tensor = torch.tensor(train_df.values.astype(np.float32))

# Parameters
input_size = data_tensor.shape[1]
learning_rate = 0.00001
num_epochs = 1000

# Model, loss function, and optimizer
model = Autoencoder(input_size)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    # Forward pass
    outputs = model(data_tensor)
    loss = criterion(outputs, data_tensor)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


Epoch [1/1000], Loss: 126613488402432000.0000
Epoch [101/1000], Loss: 126613488402432000.0000
Epoch [201/1000], Loss: 126613488402432000.0000
Epoch [301/1000], Loss: 126613488402432000.0000
Epoch [401/1000], Loss: 126613488402432000.0000
Epoch [501/1000], Loss: 126613488402432000.0000
Epoch [601/1000], Loss: 126613488402432000.0000
Epoch [701/1000], Loss: 126613488402432000.0000
Epoch [801/1000], Loss: 126613488402432000.0000
Epoch [901/1000], Loss: 126613488402432000.0000


In [206]:
torch.save(model.state_dict(), 'models/autoencoder_model.pth')


In [229]:
def infer_autoencoder_anomaly(log_string):
    log_instance = create_http_request_log_instance_from_string(log_string)
    if log_instance is None:
        return "Invalid log format"

    df = inference_preprocess_data(log_instance)
    data_tensor = torch.tensor(df.values.astype(np.float32))

    # Load the trained model
    model = Autoencoder(input_size)
    model.load_state_dict(torch.load('models/autoencoder_model.pth'))
    model.eval()

    # Predict using the autoencoder model
    reconstructed = model(data_tensor)
    loss = nn.MSELoss()(reconstructed, data_tensor)
    print("LOSS IS:",loss.item())

    # A higher reconstruction loss indicates an anomaly
    anomaly_threshold = 126795010000000000 # Define a threshold based on your understanding of the data
    if loss.item() > anomaly_threshold:
        return "Anomalous log detected"
    else:
        return "Normal log"


In [230]:
example_log = '''103.235.197.252 1631776695 "POST /dashboard/hospital/services/create" HTTP/1.0 302 513982 "-" "python-requests/2.25.1" "-" "-" - {"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"} {"available_icu_beds": "2", "available_ventilators": "2", "available_isolation_beds": "2", "csrfmiddlewaretoken": "jbf3IhTNOuFV4c8elv6oZ1vDmJU2UufyURHMB45FUl88NEKIU6aakPwOandB2RPG"} {} 
'''
result = infer_autoencoder_anomaly(example_log)
print(result)
print("\n\n\n")

103.235.197.252 1631776695 "POST /dashboard/hospital/services/create" HTTP/1.0 302 513982 "-" "python-requests/2.25.1" "-" "-" - {"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"} {"available_icu_beds": "2", "available_ventilators": "2", "available_isolation_beds": "2", "csrfmiddlewaretoken": "jbf3IhTNOuFV4c8elv6oZ1vDmJU2UufyURHMB45FUl88NEKIU6aakPwOandB2RPG"} {} 
 log
{'client_host': '103.235.197.252', 'request_time': 1631776695.0, 'request_method': 'POST', 'request_path': '/dashboard/hospital/services/create', 'http_version': 'HTTP/1.0', 'response_status': 302, 'process_time': 513982.0, 'referrer': '-', 'user_agent': 'python-requests/2.25.1', 'user_id': None, 'log_message': '-', 'cookies': '{"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"}', 'post_params': '{"available_icu_beds": "2", "available_ventilators": "2", "available_i

In [231]:
example_log = '''110.44.127.184 1631776695 "GET /dashboard/ambulance/details/create" HTTP/1.0 200 344985 "http://localhost:3000/dashboard/ambulance/details" "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/93.0.4577.82 Safari/537.36" "-" "14" - {"fastapi-csrf-token": "ImTj7Rh3vfJsG3SiX0IG5WvlvTZH6hnV5vT2OKw3RESk3Y29UyJmff5F4WyBiNFh", "sessionId": "uvl56678b8zjjtab7tignmfkhlz6c6zg"} {} {} 
'''
result = infer_autoencoder_anomaly(example_log)
print(result)
print("\n\n\n")

110.44.127.184 1631776695 "GET /dashboard/ambulance/details/create" HTTP/1.0 200 344985 "http://localhost:3000/dashboard/ambulance/details" "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/93.0.4577.82 Safari/537.36" "-" "14" - {"fastapi-csrf-token": "ImTj7Rh3vfJsG3SiX0IG5WvlvTZH6hnV5vT2OKw3RESk3Y29UyJmff5F4WyBiNFh", "sessionId": "uvl56678b8zjjtab7tignmfkhlz6c6zg"} {} {} 
 log
{'client_host': '110.44.127.184', 'request_time': 1631776695.0, 'request_method': 'GET', 'request_path': '/dashboard/ambulance/details/create', 'http_version': 'HTTP/1.0', 'response_status': 200, 'process_time': 344985.0, 'referrer': 'http://localhost:3000/dashboard/ambulance/details', 'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/93.0.4577.82 Safari/537.36', 'user_id': 14, 'log_message': '-', 'cookies': '{"fastapi-csrf-token": "ImTj7Rh3vfJsG3SiX0IG5WvlvTZH6hnV5vT2OKw3RESk3Y29UyJmff5F4WyBiNFh", "sessionId": "uvl56678b

In [232]:
example_log = '''110.44.127.184 1631776695 "POST /dashboard/accounts/login/" HTTP/1.0 200 442141 "http://logcovid.prixa.live/dashboard/accounts/login/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" __all__Incorrect username or password {"csrftoken": "FVP3hXV2nyOLBMStc6ygxpBcFyT9DZtUArYXbFvAgQ1IYReEHh7cVHoGTrI6Cl9w"} {"csrfmiddlewaretoken": "PWq66f5eR4tzNkP4n4GJnxmj4Bdq3Bd2Ksz00XFMKmGwapbfSffFLP9Niu2n2XTE", "username": "sdf", "password": "sdf))) OR ROW(2592,4840)>(SELECT COUNT(*),CONCAT(0x7162766b71,(SELECT (ELT(2592=2592,1))),0x716a767671,FLOOR(RAND(0)*2))x FROM (SELECT 8763 UNION SELECT 3269 UNION SELECT 7213 UNION SELECT 4244)a GROUP BY x) AND (((4527=4527"} {} 
'''
result = infer_autoencoder_anomaly(example_log)
print(result)
print("\n\n\n")

110.44.127.184 1631776695 "POST /dashboard/accounts/login/" HTTP/1.0 200 442141 "http://logcovid.prixa.live/dashboard/accounts/login/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" __all__Incorrect username or password {"csrftoken": "FVP3hXV2nyOLBMStc6ygxpBcFyT9DZtUArYXbFvAgQ1IYReEHh7cVHoGTrI6Cl9w"} {"csrfmiddlewaretoken": "PWq66f5eR4tzNkP4n4GJnxmj4Bdq3Bd2Ksz00XFMKmGwapbfSffFLP9Niu2n2XTE", "username": "sdf", "password": "sdf))) OR ROW(2592,4840)>(SELECT COUNT(*),CONCAT(0x7162766b71,(SELECT (ELT(2592=2592,1))),0x716a767671,FLOOR(RAND(0)*2))x FROM (SELECT 8763 UNION SELECT 3269 UNION SELECT 7213 UNION SELECT 4244)a GROUP BY x) AND (((4527=4527"} {} 
 log
{'client_host': '110.44.127.184', 'request_time': 1631776695.0, 'request_method': 'POST', 'request_path': '/dashboard/accounts/login/', 'http_version': 'HTTP/1.0', 'response_status': 200, 'process_time': 442141.0, 'referrer': 'http://logcovid.prixa.live/dashboard/accounts/login/', 'user_agen

In [237]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
import joblib

# Assuming df is your preprocessed DataFrame
dbscan = DBSCAN(eps=0.5, min_samples=5)
cluster_labels = dbscan.fit_predict(df)

# Save the labels if needed
joblib.dump(cluster_labels, 'models/dbscan_labels.pkl')

# Fit NearestNeighbors for use during inference
neigh = NearestNeighbors(n_neighbors=5)
neigh.fit(df)

# Save the NearestNeighbors model
joblib.dump(neigh, 'models/nearest_neighbors_model.pkl')


['models/nearest_neighbors_model.pkl']

In [258]:

def inference_preprocess_multi_data(log_instances):
    processed_data = []

    for log_instance in log_instances:
        log_data = log_instance.__dict__

        # Handle JSON fields
        json_features = ['cookies', 'post_params', 'get_params']
        for field in json_features:
            log_data[field] = json.loads(log_data[field]) if log_data[field] else {}
            log_data[f"{field}_count"] = len(log_data[field].keys())

        # Handling textual data (e.g., user_agent)
        log_data['browser'] = httpagentparser.detect(log_data['user_agent'])['browser']['name'] if httpagentparser.detect(log_data['user_agent']).get('browser') else 'Unknown'

        # Add processed log data to list
        processed_data.append(log_data)

    # Create a DataFrame from the processed data
    df = pd.DataFrame(processed_data)
    df = df.drop(['_sa_instance_state'], axis=1)

    # Fill missing categorical columns with 'Unknown'
    categorical_features = ['request_method', 'http_version', 'user_id', 'browser']
    for feature in categorical_features:
        if feature not in df.columns:
            df[feature] = 'Unknown'

    # Apply one-hot encoding to the categorical features
    encoded_df = pd.DataFrame(one_hot_encoder.transform(df[categorical_features]).toarray(), columns=one_hot_encoder.get_feature_names_out())

    # Concatenate encoded features with the rest of the data
    df = pd.concat([df.drop(columns=categorical_features), encoded_df], axis=1)

    # Drop the original categorical and JSON columns as they are now encoded
    fields_to_drop = json_features + ['user_agent', 'referrer', 'log_message', 'body', 'client_host', 'request_path']
    df = df.drop(columns=fields_to_drop)

    return df

In [293]:
def infer_dbscan_anomaly(log_string_list):
    log_instances = [create_http_request_log_instance_from_string(log_string) for log_string in log_string_list]
    if not isinstance(log_instances, list):
        log_instances = [log_instances]

    # Assuming log_instances is a list of log_instance objects
    # Preprocess each instance and create a DataFrame
    df = pd.concat([inference_preprocess_data(instance) for instance in log_instances])

    # Reset the index of the DataFrame
    df.reset_index(drop=True, inplace=True)

    # Load the NearestNeighbors model
    neigh = joblib.load('models/nearest_neighbors_model.pkl')


    # Number of neighbors for comparison
    n_neighbors = min(len(df), neigh.n_samples_fit_)  # Ensure n_neighbors does not exceed the number of samples

    # Check the distance to the nearest neighbors in the original dataset
    distances, indices = neigh.kneighbors(df, n_neighbors=n_neighbors)

    # Define a threshold for anomaly detection
    distance_threshold = 1601111  # Adjust based on your data

    # Check if the mean distance is greater than the threshold
    results = []
    for row in distances:
        if np.mean(row) > distance_threshold:
            results.append("Anomalous log detected")
        else:
            results.append("Normal log")
    return results

In [294]:
example_log = ['''103.235.197.252 1631776695 "POST /dashboard/hospital/services/create" HTTP/1.0 302 513982 "-" "python-requests/2.25.1" "-" "-" - {"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"} {"available_icu_beds": "2", "available_ventilators": "2", "available_isolation_beds": "2", "csrfmiddlewaretoken": "jbf3IhTNOuFV4c8elv6oZ1vDmJU2UufyURHMB45FUl88NEKIU6aakPwOandB2RPG"} {} ''']
result = infer_dbscan_anomaly(example_log)
print(result)
print("\n\n\n")

example_log = ['''110.44.127.184 1631776695 "GET /dashboard/ambulance/details/create" HTTP/1.0 200 344985 "http://localhost:3000/dashboard/ambulance/details" "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/93.0.4577.82 Safari/537.36" "-" "14" - {"fastapi-csrf-token": "ImTj7Rh3vfJsG3SiX0IG5WvlvTZH6hnV5vT2OKw3RESk3Y29UyJmff5F4WyBiNFh", "sessionId": "uvl56678b8zjjtab7tignmfkhlz6c6zg"} {} {} 
''']
result = infer_dbscan_anomaly(example_log)
print(result)
print("\n\n\n")

example_log = ['''110.44.127.184 1631776695 "POST /dashboard/accounts/login/" HTTP/1.0 200 442141 "http://logcovid.prixa.live/dashboard/accounts/login/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" __all__Incorrect username or password {"csrftoken": "FVP3hXV2nyOLBMStc6ygxpBcFyT9DZtUArYXbFvAgQ1IYReEHh7cVHoGTrI6Cl9w"} {"csrfmiddlewaretoken": "PWq66f5eR4tzNkP4n4GJnxmj4Bdq3Bd2Ksz00XFMKmGwapbfSffFLP9Niu2n2XTE", "username": "sdf", "password": "sdf))) OR ROW(2592,4840)>(SELECT COUNT(*),CONCAT(0x7162766b71,(SELECT (ELT(2592=2592,1))),0x716a767671,FLOOR(RAND(0)*2))x FROM (SELECT 8763 UNION SELECT 3269 UNION SELECT 7213 UNION SELECT 4244)a GROUP BY x) AND (((4527=4527"} {} 
''']
result = infer_dbscan_anomaly(example_log)
print(result)
print("\n\n\n")

103.235.197.252 1631776695 "POST /dashboard/hospital/services/create" HTTP/1.0 302 513982 "-" "python-requests/2.25.1" "-" "-" - {"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"} {"available_icu_beds": "2", "available_ventilators": "2", "available_isolation_beds": "2", "csrfmiddlewaretoken": "jbf3IhTNOuFV4c8elv6oZ1vDmJU2UufyURHMB45FUl88NEKIU6aakPwOandB2RPG"} {}  log
{'client_host': '103.235.197.252', 'request_time': 1631776695.0, 'request_method': 'POST', 'request_path': '/dashboard/hospital/services/create', 'http_version': 'HTTP/1.0', 'response_status': 302, 'process_time': 513982.0, 'referrer': '-', 'user_agent': 'python-requests/2.25.1', 'user_id': None, 'log_message': '-', 'cookies': '{"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"}', 'post_params': '{"available_icu_beds": "2", "available_ventilators": "2", "available_is

In [295]:
example_log = ['''103.235.197.252 1631776695 "POST /dashboard/hospital/services/create" HTTP/1.0 302 513982 "-" "python-requests/2.25.1" "-" "-" - {"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"} {"available_icu_beds": "2", "available_ventilators": "2", "available_isolation_beds": "2", "csrfmiddlewaretoken": "jbf3IhTNOuFV4c8elv6oZ1vDmJU2UufyURHMB45FUl88NEKIU6aakPwOandB2RPG"} {} ''',
              '''110.44.127.184 1631776695 "GET /dashboard/ambulance/details/create" HTTP/1.0 200 344985 "http://localhost:3000/dashboard/ambulance/details" "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/93.0.4577.82 Safari/537.36" "-" "14" - {"fastapi-csrf-token": "ImTj7Rh3vfJsG3SiX0IG5WvlvTZH6hnV5vT2OKw3RESk3Y29UyJmff5F4WyBiNFh", "sessionId": "uvl56678b8zjjtab7tignmfkhlz6c6zg"} {} {} ''',
              '''110.44.127.184 1631776695 "POST /dashboard/accounts/login/" HTTP/1.0 200 442141 "http://logcovid.prixa.live/dashboard/accounts/login/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" __all__Incorrect username or password {"csrftoken": "FVP3hXV2nyOLBMStc6ygxpBcFyT9DZtUArYXbFvAgQ1IYReEHh7cVHoGTrI6Cl9w"} {"csrfmiddlewaretoken": "PWq66f5eR4tzNkP4n4GJnxmj4Bdq3Bd2Ksz00XFMKmGwapbfSffFLP9Niu2n2XTE", "username": "sdf", "password": "sdf))) OR ROW(2592,4840)>(SELECT COUNT(*),CONCAT(0x7162766b71,(SELECT (ELT(2592=2592,1))),0x716a767671,FLOOR(RAND(0)*2))x FROM (SELECT 8763 UNION SELECT 3269 UNION SELECT 7213 UNION SELECT 4244)a GROUP BY x) AND (((4527=4527"} {}''',
               '''103.235.197.252 1631776695 "POST /dashboard/hospital/services/create" HTTP/1.0 302 514074 "-" "python-requests/2.25.1" "-" "-" - {"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"} {"available_icu_beds": "2", "available_ventilators": "2", "available_isolation_beds": "2", "csrfmiddlewaretoken": "jbf3IhTNOuFV4c8elv6oZ1vDmJU2UufyURHMB45FUl88NEKIU6aakPwOandB2RPG"} {} ''',
               '''103.235.197.252 1630333830 "GET /?search=-9505%20OR%204645%3D4645%23%20bqim&district=47" HTTP/1.0 200 1957489 "http://logcovid.prixa.live:80/" "sqlmap/1.5.9#stable (http://sqlmap.org)" "-" "-" - {"csrftoken": "9HfLGqyoEYlD1UImv94ImZsYrxTAjC4v8VwZHmRkOrxH8dtmsWTyDYOtFufkTX7B"} {} {"search": "-9505 OR 4645=4645# bqim", "district": "47"} '''

]
result = infer_dbscan_anomaly(example_log)
print(result)
print("\n\n\n")

# example_log = ['''110.44.127.184 1631776695 "GET /dashboard/ambulance/details/create" HTTP/1.0 200 344985 "http://localhost:3000/dashboard/ambulance/details" "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/93.0.4577.82 Safari/537.36" "-" "14" - {"fastapi-csrf-token": "ImTj7Rh3vfJsG3SiX0IG5WvlvTZH6hnV5vT2OKw3RESk3Y29UyJmff5F4WyBiNFh", "sessionId": "uvl56678b8zjjtab7tignmfkhlz6c6zg"} {} {} 
# ''']
# result = infer_dbscan_anomaly(example_log)
# print(result)
# print("\n\n\n")

# example_log = ['''110.44.127.184 1631776695 "POST /dashboard/accounts/login/" HTTP/1.0 200 442141 "http://logcovid.prixa.live/dashboard/accounts/login/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" __all__Incorrect username or password {"csrftoken": "FVP3hXV2nyOLBMStc6ygxpBcFyT9DZtUArYXbFvAgQ1IYReEHh7cVHoGTrI6Cl9w"} {"csrfmiddlewaretoken": "PWq66f5eR4tzNkP4n4GJnxmj4Bdq3Bd2Ksz00XFMKmGwapbfSffFLP9Niu2n2XTE", "username": "sdf", "password": "sdf))) OR ROW(2592,4840)>(SELECT COUNT(*),CONCAT(0x7162766b71,(SELECT (ELT(2592=2592,1))),0x716a767671,FLOOR(RAND(0)*2))x FROM (SELECT 8763 UNION SELECT 3269 UNION SELECT 7213 UNION SELECT 4244)a GROUP BY x) AND (((4527=4527"} {} 
# ''']
# result = infer_dbscan_anomaly(example_log)
# print(result)
# print("\n\n\n")

103.235.197.252 1631776695 "POST /dashboard/hospital/services/create" HTTP/1.0 302 513982 "-" "python-requests/2.25.1" "-" "-" - {"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"} {"available_icu_beds": "2", "available_ventilators": "2", "available_isolation_beds": "2", "csrfmiddlewaretoken": "jbf3IhTNOuFV4c8elv6oZ1vDmJU2UufyURHMB45FUl88NEKIU6aakPwOandB2RPG"} {}  log
{'client_host': '103.235.197.252', 'request_time': 1631776695.0, 'request_method': 'POST', 'request_path': '/dashboard/hospital/services/create', 'http_version': 'HTTP/1.0', 'response_status': 302, 'process_time': 513982.0, 'referrer': '-', 'user_agent': 'python-requests/2.25.1', 'user_id': None, 'log_message': '-', 'cookies': '{"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"}', 'post_params': '{"available_icu_beds": "2", "available_ventilators": "2", "available_is

In [252]:
from sklearn.mixture import GaussianMixture

# Assuming df is your preprocessed DataFrame
gmm = GaussianMixture(n_components=2, covariance_type='full')
gmm.fit(train_df)


GaussianMixture(n_components=2)

In [253]:
import joblib

# Save the model
joblib.dump(gmm, 'models/gussian_mix_model.pkl')

['models/gussian_mix_model.pkl']

In [256]:
def infer_gmm_anomaly(log_string):
    log_instance = create_http_request_log_instance_from_string(log_string)
    if log_instance is None:
        return "Invalid log format"

    df = inference_preprocess_data(log_instance)

    # Load the trained GMM model
    gmm = joblib.load('models/gussian_mix_model.pkl')

    # Compute log likelihood
    log_likelihood = gmm.score_samples(df)

    # Define a threshold for anomaly detection based on your data
    anomaly_threshold = 0.5  # This needs to be determined based on your data
    print(log_likelihood,1111)

    if log_likelihood < anomaly_threshold:
        return "Anomalous log detected"
    else:
        return "Normal log"


In [257]:
example_log = '''103.235.197.252 1631776695 "POST /dashboard/hospital/services/create" HTTP/1.0 302 513982 "-" "python-requests/2.25.1" "-" "-" - {"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"} {"available_icu_beds": "2", "available_ventilators": "2", "available_isolation_beds": "2", "csrfmiddlewaretoken": "jbf3IhTNOuFV4c8elv6oZ1vDmJU2UufyURHMB45FUl88NEKIU6aakPwOandB2RPG"} {} 
'''
result = infer_gmm_anomaly(example_log)
print(result)
print("\n\n\n")

example_log = '''110.44.127.184 1631776695 "GET /dashboard/ambulance/details/create" HTTP/1.0 200 344985 "http://localhost:3000/dashboard/ambulance/details" "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/93.0.4577.82 Safari/537.36" "-" "14" - {"fastapi-csrf-token": "ImTj7Rh3vfJsG3SiX0IG5WvlvTZH6hnV5vT2OKw3RESk3Y29UyJmff5F4WyBiNFh", "sessionId": "uvl56678b8zjjtab7tignmfkhlz6c6zg"} {} {} 
'''
result = infer_gmm_anomaly(example_log)
print(result)
print("\n\n\n")

example_log = '''110.44.127.184 1631776695 "POST /dashboard/accounts/login/" HTTP/1.0 200 442141 "http://logcovid.prixa.live/dashboard/accounts/login/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:92.0) Gecko/20100101 Firefox/92.0" "-" "-" __all__Incorrect username or password {"csrftoken": "FVP3hXV2nyOLBMStc6ygxpBcFyT9DZtUArYXbFvAgQ1IYReEHh7cVHoGTrI6Cl9w"} {"csrfmiddlewaretoken": "PWq66f5eR4tzNkP4n4GJnxmj4Bdq3Bd2Ksz00XFMKmGwapbfSffFLP9Niu2n2XTE", "username": "sdf", "password": "sdf))) OR ROW(2592,4840)>(SELECT COUNT(*),CONCAT(0x7162766b71,(SELECT (ELT(2592=2592,1))),0x716a767671,FLOOR(RAND(0)*2))x FROM (SELECT 8763 UNION SELECT 3269 UNION SELECT 7213 UNION SELECT 4244)a GROUP BY x) AND (((4527=4527"} {} 
'''
result = infer_gmm_anomaly(example_log)
print(result)
print("\n\n\n")

103.235.197.252 1631776695 "POST /dashboard/hospital/services/create" HTTP/1.0 302 513982 "-" "python-requests/2.25.1" "-" "-" - {"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"} {"available_icu_beds": "2", "available_ventilators": "2", "available_isolation_beds": "2", "csrfmiddlewaretoken": "jbf3IhTNOuFV4c8elv6oZ1vDmJU2UufyURHMB45FUl88NEKIU6aakPwOandB2RPG"} {} 
 log
{'client_host': '103.235.197.252', 'request_time': 1631776695.0, 'request_method': 'POST', 'request_path': '/dashboard/hospital/services/create', 'http_version': 'HTTP/1.0', 'response_status': 302, 'process_time': 513982.0, 'referrer': '-', 'user_agent': 'python-requests/2.25.1', 'user_id': None, 'log_message': '-', 'cookies': '{"csrftoken": "F7H2NHrT5yTYUGvm1p6xCIgZrEv3mPz4gN9LGuDLbpmbD87QA0ajXwhafiOCuc9c", "sessionid": "a9xo8ijwwy9cxy9jn3u7pg55sp4z6lip"}', 'post_params': '{"available_icu_beds": "2", "available_ventilators": "2", "available_i